# Connect Python to MySQL

In [2]:
import mysql.connector
import pandas as pd
import random
from datetime import date, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root"
)

if conn.is_connected():
    print("Connected to MySQL")

cursor = conn.cursor()

Connected to MySQL


# Create Database

In [7]:
cursor.execute("CREATE DATABASE IF NOT EXISTS healthcare_analytics")
cursor.execute("USE healthcare_analytics")

# Medicines Table

In [9]:
cursor.execute("""
CREATE TABLE medicines (
    medicine_id INT AUTO_INCREMENT PRIMARY KEY,
    medicine_name VARCHAR(100),
    category VARCHAR(50),
    price DECIMAL(10,2),
    expiry_date DATE,
    stock_quantity INT
)
""")

# Create Tables

In [6]:
cursor.execute("""
CREATE TABLE customers (
    customer_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_name VARCHAR(100),
    city VARCHAR(50),
    join_date DATE
)
""")

ProgrammingError: 1050 (42S01): Table 'customers' already exists

# Sales Table

In [11]:
cursor.execute("""
CREATE TABLE sales (
    sale_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT,
    medicine_id INT,
    quantity INT,
    sale_date DATE,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (medicine_id) REFERENCES medicines(medicine_id)
)
""")

conn.commit()

# Insert Medicines

In [26]:
medicines = [
    ("Paracetamol", "Pain Relief", 50, "2026-12-01", 500),
    ("Vitamin C", "Vitamins", 120, "2026-10-10", 300),
    ("Insulin", "Diabetes", 500, "2026-08-20", 150),
    ("Cough Syrup", "Cough & Cold", 150, "2026-09-15", 250),
    ("BP Tablet", "Blood Pressure", 200, "2026-11-05", 200)
]

cursor.executemany("""
INSERT INTO medicines 
(medicine_name, category, price, expiry_date, stock_quantity)
VALUES (%s, %s, %s, %s, %s)
""", medicines)

conn.commit()

# Insert Customers

In [13]:
cities = ["Mumbai", "Pune", "Delhi", "Bangalore"]
customers = []

for i in range(20):
    customers.append((
        f"Customer_{i+1}",
        random.choice(cities),
        date(2024,1,1) + timedelta(days=random.randint(1,200))
    ))

cursor.executemany("""
INSERT INTO customers (customer_name, city, join_date)
VALUES (%s, %s, %s)
""", customers)

conn.commit()

# Insert Sales Transactions


In [14]:
sales = []

for _ in range(100):
    sales.append((
        random.randint(1,20),
        random.randint(1,5),
        random.randint(1,20),
        date(2024,1,1) + timedelta(days=random.randint(1,300))
    ))

cursor.executemany("""
INSERT INTO sales (customer_id, medicine_id, quantity, sale_date)
VALUES (%s, %s, %s, %s)
""", sales)

conn.commit()

# Total Revenue

In [15]:
query = """
SELECT SUM(s.quantity * m.price) AS total_revenue
FROM sales s
JOIN medicines m ON s.medicine_id = m.medicine_id
"""

revenue_df = pd.read_sql(query, conn)
revenue_df

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\3300109065.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  revenue_df = pd.read_sql(query, conn)


,total_revenue
0,221020.0


# Category-wise Sales

In [16]:
query = """
SELECT m.category,
       SUM(s.quantity * m.price) AS category_revenue
FROM sales s
JOIN medicines m ON s.medicine_id = m.medicine_id
GROUP BY m.category
"""

category_df = pd.read_sql(query, conn)
category_df

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\1110663802.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  category_df = pd.read_sql(query, conn)


,category,category_revenue
0,Pain Relief,9200.0
1,Vitamins,34920.0
2,Diabetes,104000.0
3,Cough & Cold,25500.0
4,Blood Pressure,47400.0


In [18]:
"""
SELECT m.medicine_name,
       SUM(s.quantity) AS total_quantity
FROM sales s
JOIN medicines m ON s.medicine_id = m.medicine_id
GROUP BY m.medicine_name
ORDER BY total_quantity DESC
LIMIT 1
"""

top_medicine_df = pd.read_sql(query, conn)
top_medicine_df

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\20061284.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_medicine_df = pd.read_sql(query, conn)


,category,category_revenue
0,Pain Relief,9200.0
1,Vitamins,34920.0
2,Diabetes,104000.0
3,Cough & Cold,25500.0
4,Blood Pressure,47400.0


# Top Selling Medicine

In [19]:
query = """
SELECT m.medicine_name,
       SUM(s.quantity) AS total_quantity
FROM sales s
JOIN medicines m ON s.medicine_id = m.medicine_id
GROUP BY m.medicine_name
ORDER BY total_quantity DESC
LIMIT 1
"""

top_medicine_df = pd.read_sql(query, conn)
top_medicine_df

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\2752093825.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_medicine_df = pd.read_sql(query, conn)


,medicine_name,total_quantity
0,Vitamin C,291.0


# Monthly Sales Trend

In [20]:
query = """
SELECT DATE_FORMAT(s.sale_date,'%Y-%m') AS month,
       SUM(s.quantity * m.price) AS revenue
FROM sales s
JOIN medicines m ON s.medicine_id = m.medicine_id
GROUP BY month
ORDER BY month
"""

monthly_df = pd.read_sql(query, conn)
monthly_df["growth_%"] = monthly_df["revenue"].pct_change()*100
monthly_df

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\4153434539.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  monthly_df = pd.read_sql(query, conn)


,month,revenue,growth_%
0,2024-01,11760.0,NaN
1,2024-02,16890.0,43.622449
2,2024-03,17540.0,3.848431
3,2024-04,24640.0,40.478905
4,2024-05,33580.0,36.282468
5,2024-06,29210.0,-13.013699
6,2024-07,38930.0,33.276275
7,2024-08,26950.0,-30.773183
8,2024-09,17480.0,-35.139147
9,2024-10,4040.0,-76.887872


# High Value Customers

In [21]:
high_value = pd.read_sql("""
SELECT s.customer_id,
       SUM(s.quantity * m.price) AS total_spent
FROM sales s
JOIN medicines m ON s.medicine_id = m.medicine_id
GROUP BY s.customer_id
HAVING total_spent > 50000
""", conn)

high_value

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\365899321.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  high_value = pd.read_sql("""


,customer_id,total_spent


# Expiry Alert (Next 60 Days)

In [22]:
expiry_alert = pd.read_sql("""
SELECT medicine_name, expiry_date
FROM medicines
WHERE expiry_date < CURDATE() + INTERVAL 60 DAY
""", conn)

expiry_alert

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\3801607260.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  expiry_alert = pd.read_sql("""


,medicine_name,expiry_date


# KPI Dashboard

In [23]:
kpi = {}

kpi["Total Customers"] = pd.read_sql(
    "SELECT COUNT(*) FROM customers", conn
).iloc[0,0]

kpi["Total Sales Transactions"] = pd.read_sql(
    "SELECT COUNT(*) FROM sales", conn
).iloc[0,0]

kpi["Total Revenue"] = revenue_df.iloc[0,0]

pd.DataFrame(kpi.items(), columns=["KPI", "Value"])

C:\Users\PC\AppData\Local\Temp\ipykernel_13568\1862869233.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  kpi["Total Customers"] = pd.read_sql(
C:\Users\PC\AppData\Local\Temp\ipykernel_13568\1862869233.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  kpi["Total Sales Transactions"] = pd.read_sql(


,KPI,Value
0,Total Customers,20.0
1,Total Sales Transactions,100.0
2,Total Revenue,221020.0


# Export for Power BI

In [25]:
category_df.to_csv("category.csv", index=False)
monthly_df.to_csv("monthly.csv", index=False)
top_medicine_df.to_csv("top_medicine.csv", index=False)
high_value.to_csv("high_value_customers.csv", index=False)